# カオス 2：ローレンツ系 ―ストレンジアトラクタと、3つのリアプノフ指数

1963年に気象学者ローレンツが、大気の対流を極端に簡単にしたモデルとして調べた微分方程式です：

$$
\dot x=\sigma(y-x),\qquad \dot y=x(\rho-z)-y,\qquad \dot z=xy-\beta z
$$

標準的なパラメータ $\sigma=10,\ \rho=28,\ \beta=8/3$ で、カオスになります。ロジスティック写像（1冊目）との違いは、**連続時間**で**3次元**であることです。差の広がり方は、1次元の「傾き」の代わりに**ヤコビ行列**で決まり、リアプノフ指数は3つになります。

[学習ロードマップ](../../../docs/learning_roadmap.md)の候補14、ノートブック先行。各節の「考えてみよう」は、後でノートにまとめる材料です。

In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as W
from numpy.typing import NDArray
from scipy.integrate import solve_ivp

Floats = NDArray[np.float64]   # 実数の配列（型ヒント用の別名）
# 注意：FigureWidget（スライダーで描き直す図）には、numpy 配列ではなくリストを渡す（NB-02 を参照）

SIGMA, RHO, BETA = 10.0, 28.0, 8.0 / 3.0


def lorenz(t: float, v: Floats, sigma: float = SIGMA, rho: float = RHO, beta: float = BETA) -> Floats:
    x, y, z = v
    return np.array([sigma * (y - x), x * (rho - z) - y, x * y - beta * z])


def jacobian(v: Floats, sigma: float = SIGMA, rho: float = RHO, beta: float = BETA) -> Floats:
    """ローレンツ系のヤコビ行列 ∂(ẋ, ẏ, ż)/∂(x, y, z)"""
    x, y, z = v
    return np.array([[-sigma, sigma, 0.0], [rho - z, -1.0, -x], [y, x, -beta]])


def trajectory(v0: Floats, T: float, rho: float = RHO, n: int = 8000) -> Floats:
    """時刻 0〜T の軌道（行が時刻、列が x, y, z）"""
    sol = solve_ivp(lorenz, (0, T), v0, args=(SIGMA, rho, BETA), t_eval=np.linspace(0, T, n),
                    rtol=1e-10, atol=1e-12)
    return sol.y.T

## 1. ストレンジアトラクタ

$(1,1,1)$ から $t=50$ まで追った軌道を、3次元で描きます（マウスで回転できます）。

In [ ]:
P = trajectory(np.array([1.0, 1.0, 1.0]), 50.0)
fig = go.Figure(go.Scatter3d(x=P[:, 0].tolist(), y=P[:, 1].tolist(), z=P[:, 2].tolist(), mode="lines",
                             line=dict(color=np.linspace(0, 1, len(P)).tolist(), colorscale="Viridis", width=2)))
fig.update_layout(height=550, width=700, margin=dict(l=0, r=0, t=10, b=0),
                  scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"))
fig

**考えてみよう**

- 軌道は2つの「目」のまわりを回り、ときどき反対側に乗り移ります。乗り移るタイミングに、規則は見えるでしょうか。
- 軌道は決して交わりません（同じ点からは同じ未来しかない、という微分方程式の性質）。それなのに、有限の範囲に永遠に収まっています。どうしてそんなことが可能なのでしょうか（§4 の体積の縮みと、§1 の折りたたまれた形がヒントです）。

## 2. 近い2つの軌道は、指数関数的に離れる

アトラクタの上の点（$(1,1,1)$ から $t=20$ まで進めた点）と、そこから $10^{-9}$ だけずらした点の2つから始め、距離を対数目盛りで描きます。直線の部分の傾きが、最大のリアプノフ指数 $\lambda_1$ です（文献値は約 $0.906$）。

In [ ]:
d0 = 1e-9
v0 = trajectory(np.array([1.0, 1.0, 1.0]), 20.0)[-1]   # アトラクタの上の点から始める
Pa = trajectory(v0, 40.0)
Pb = trajectory(v0 + np.array([d0, 0, 0]), 40.0)
t = np.linspace(0, 40.0, len(Pa))
dist = np.linalg.norm(Pa - Pb, axis=1)

mask = (t > 1) & (dist < 1e-1)
slope = np.polyfit(t[mask], np.log(dist[mask]), 1)[0]
print(f"傾き（λ₁ の推定）= {slope:.3f}（文献値 ≈ 0.906）")
assert 0.6 < slope < 1.2

fig = go.Figure(go.Scatter(x=t.tolist(), y=dist.tolist(), mode="lines", name="距離"))
fig.update_layout(height=360, width=750, yaxis_type="log", xaxis_title="時刻 t", yaxis_title="2つの軌道の距離",
                  margin=dict(t=20))
fig

## 3. 3つのリアプノフ指数（ベネッティンの方法）

小さな差 $\delta\mathbf v$ は、ヤコビ行列 $J$ で $\dot{\delta\mathbf v}=J(\mathbf v(t))\,\delta\mathbf v$ と運ばれます。3つの方向の差をまとめて行列 $Q$ に入れて運び、ときどき **QR 分解**で直交化し直します（そのままだと、すべての方向が最も伸びる方向に揃ってしまうため）。$R$ の対角成分の対数の平均が、3つのリアプノフ指数 $\lambda_1\ge\lambda_2\ge\lambda_3$ です。

In [ ]:
def lyapunov_spectrum(v0: Floats, dt: float = 0.01, steps: int = 60000, renorm: int = 10,
                      transient: int = 2000) -> Floats:
    """軌道と接ベクトルを同時に 4 次のルンゲ・クッタで進め、QR 分解で直交化しながら指数を求める"""
    def rhs(v: Floats, Q: Floats) -> tuple[Floats, Floats]:
        return lorenz(0.0, v), jacobian(v) @ Q

    v, Q = v0.astype(float), np.eye(3)
    for _ in range(transient):                        # まずアトラクタの上に乗せる
        k1 = lorenz(0, v); k2 = lorenz(0, v + dt / 2 * k1); k3 = lorenz(0, v + dt / 2 * k2); k4 = lorenz(0, v + dt * k3)
        v = v + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    sums = np.zeros(3)
    for k in range(steps):
        a1 = rhs(v, Q)
        a2 = rhs(v + dt / 2 * a1[0], Q + dt / 2 * a1[1])
        a3 = rhs(v + dt / 2 * a2[0], Q + dt / 2 * a2[1])
        a4 = rhs(v + dt * a3[0], Q + dt * a3[1])
        v = v + dt / 6 * (a1[0] + 2 * a2[0] + 2 * a3[0] + a4[0])
        Q = Q + dt / 6 * (a1[1] + 2 * a2[1] + 2 * a3[1] + a4[1])
        if (k + 1) % renorm == 0:
            Q, R = np.linalg.qr(Q)
            sums += np.log(np.abs(np.diag(R)))
    return sums / (steps * dt)


lams = lyapunov_spectrum(np.array([1.0, 1.0, 1.0]))
print("リアプノフ指数 (λ₁, λ₂, λ₃) =", np.round(lams, 3), "（文献値 ≈ 0.906, 0, −14.57）")
print(f"指数の和 = {lams.sum():.4f}、−(σ + 1 + β) = {-(SIGMA + 1 + BETA):.4f}")
assert abs(lams[0] - 0.906) < 0.05 and abs(lams[1]) < 0.05
assert abs(lams.sum() + (SIGMA + 1 + BETA)) < 1e-2

**考えてみよう**

- 3つの指数は「正・ゼロ・負」です。正は「差が広がる方向」、負は「アトラクタに吸い寄せられる方向」です。ゼロの方向は何でしょうか（ヒント：同じ軌道の上で、時刻を少しずらした点との差）。
- 指数の和が $-(\sigma+1+\beta)$ にぴったり一致しました。次の節で理由を見ます。

## 4. 体積は縮み続ける：指数の和 = 発散

3つの方向の差が張る小さな箱の体積は、$\det Q$ に比例し、$\frac{d}{dt}\ln\det Q=\operatorname{tr}J$ で変わります（[det のノート](../../../研究ノート/04_群論・代数/det_exp_trace_proof.md)の Jacobi の公式と補足2「トレースは体積の変化率」）。ローレンツ系のヤコビ行列のトレース（ベクトル場の発散）は、場所によらず

$$
\operatorname{tr}J=-\sigma-1-\beta=-13.67
$$

なので、体積は $e^{-13.67\,t}$ で縮みます。リアプノフ指数の和は、この縮む速さそのものです（[リー微分のノート](../../../研究ノート/02_微分幾何/lie_derivative.md)の Part VIII「発散＝体積の変化率」の非線形版）。

下のセルで、アトラクタ上の点のまわりの小さな立方体（3本の辺）を運び、辺が張る平行六面体の体積（行列式）が縮む様子を、短い時間で確かめます。

In [ ]:
# 小さな立方体の8つの頂点を運び、体積（平行六面体）の変化を測る
h = 1e-6
base = trajectory(np.array([1.0, 1.0, 1.0]), 20.0)[-1]          # アトラクタの上の点
edges = np.eye(3) * h
T_short, n_t = 0.3, 61
ts = np.linspace(0, T_short, n_t)
sol0 = solve_ivp(lorenz, (0, T_short), base, t_eval=ts, rtol=1e-12, atol=1e-14).y.T
vols = []
cols = [solve_ivp(lorenz, (0, T_short), base + e, t_eval=ts, rtol=1e-12, atol=1e-14).y.T for e in edges]
for k in range(n_t):
    M = np.stack([c[k] - sol0[k] for c in cols], axis=1)
    vols.append(abs(np.linalg.det(M)))
vols = np.array(vols) / h**3
rate = np.polyfit(ts, np.log(vols), 1)[0]
print(f"体積の対数の傾き = {rate:.3f}、tr J = −(σ+1+β) = {-(SIGMA + 1 + BETA):.3f}")
assert abs(rate + (SIGMA + 1 + BETA)) < 0.1

fig = go.Figure([go.Scatter(x=ts.tolist(), y=vols.tolist(), mode="lines+markers", name="体積（計算）"),
                 go.Scatter(x=ts.tolist(), y=np.exp(-(SIGMA + 1 + BETA) * ts).tolist(), mode="lines",
                            line=dict(dash="dash"), name="e^{tr J · t}")])
fig.update_layout(height=360, width=750, yaxis_type="log", xaxis_title="時刻 t", yaxis_title="体積（最初を 1 とする）",
                  margin=dict(t=20))
fig

**考えてみよう**

- 体積は縮み続けるのに、ある方向（$\lambda_1>0$）には伸び続けます。伸びながら全体としては縮むので、形は薄く引き伸ばされ、折りたたまれていきます。§1 のアトラクタの形を、この見方で眺め直してみましょう。
- ハミルトン系（摩擦のない力学系）では体積が保存され（リウヴィルの定理）、指数の和は $0$ になります。ローレンツ系の $\sigma,\ 1,\ \beta$ は、何に相当するでしょうか。

## 5. スライダー：$\rho$ を変えると

$\rho$（温度差の強さ）を変えると、振る舞いが変わります。

- $\rho<1$：原点に落ち着く
- $1<\rho<24.74$ くらい：2つの「目」の中心（不動点 $C_\pm$）のどちらかに、渦を巻きながら落ち着く（$\rho$ が 13.9 を超えると、しばらく迷走してから落ち着く）
- $\rho=28$：カオス

In [ ]:
fig_rho = go.FigureWidget(go.Scatter3d(x=[], y=[], z=[], mode="lines", line=dict(color="#1f77b4", width=2)))
fig_rho.add_trace(go.Scatter3d(x=[], y=[], z=[], mode="markers", marker=dict(size=4, color="#d62728"), name="不動点 C±"))
fig_rho.update_layout(height=520, width=650, margin=dict(l=0, r=0, t=10, b=0), showlegend=False,
                  scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z",
                             xaxis=dict(range=[-25, 25]), yaxis=dict(range=[-30, 30]), zaxis=dict(range=[0, 55])))
rho_slider = W.FloatSlider(value=28.0, min=0.5, max=35.0, step=0.5, description="ρ")
info = W.HTML()


def update(*_: object) -> None:
    """スライダーが動くたびに呼ばれ、軌道を計算し直す"""
    rho = rho_slider.value
    P = trajectory(np.array([1.0, 1.0, 1.0]), 40.0, rho=rho, n=4000)
    with fig_rho.batch_update():
        fig_rho.data[0].x, fig_rho.data[0].y, fig_rho.data[0].z = P[:, 0].tolist(), P[:, 1].tolist(), P[:, 2].tolist()
        if rho > 1:
            c = float(np.sqrt(BETA * (rho - 1)))
            fig_rho.data[1].x, fig_rho.data[1].y, fig_rho.data[1].z = [c, -c], [c, -c], [rho - 1, rho - 1]
        else:
            fig_rho.data[1].x, fig_rho.data[1].y, fig_rho.data[1].z = [0.0], [0.0], [0.0]
    end = P[-1]
    info.value = f"ρ = {rho}：最後の点 ({end[0]:+.2f}, {end[1]:+.2f}, {end[2]:+.2f})"


rho_slider.observe(update, "value")
update()
W.VBox([rho_slider, info, fig_rho])

## 観察のポイント（まとめ）

- 連続時間の3次元の系でも、近い軌道は指数関数的に離れます（$\lambda_1\approx0.906$）。
- 差の広がりはヤコビ行列で決まり、3つの方向の伸び縮みが、3つのリアプノフ指数になります。そのうち1つはゼロ（軌道に沿った方向）です。
- 指数の和は、ヤコビ行列のトレース（発散）の平均＝体積が縮む速さで、ローレンツ系では $-(\sigma+1+\beta)$ にぴったり一致しました。行列の「トレース＝体積の変化率」（det のノート）が、カオスの解析でも中心的な役割を果たしています。
- 伸びる方向と縮む方向があるので、軌道は有限の範囲に収まったまま、引き伸ばされて折りたたまれ、複雑な形（ストレンジアトラクタ）を作ります。

**この後**：これらの観察を、ノート（リアプノフ指数の定義、ヤコビ行列による線形化、体積の変化）にまとめます。